# BERT pairwise con particiones equilibradas

En este notebook se revisa la estrategia de particionado utilizada para el modelo BERT por pares. El objetivo es conseguir que train, los folds de validación y test tengan una composición aproximadamente comparable en cuanto a:

- Presencia de empates en el máximo número de upvotes.
- Número de respuestas candidatas por pregunta.

Todas las divisiones se realizan por questionID, manteniendo siempre juntas todas las respuestas de una misma pregunta.

Una vez comprobadas las nuevas particiones, se repetirá el entrenamiento y evaluación del modelo BERT pairwise.

In [1]:
import sys
print(sys.executable)

/usr/bin/python3


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

In [3]:
url = (
    "https://huggingface.co/datasets/nbertagnolli/counsel-chat/"
    "raw/main/20220401_counsel_chat.csv"
)

df = pd.read_csv(url)

print("Preguntas totales:", df["questionID"].nunique())
print("Respuestas totales:", len(df))

Preguntas totales: 940
Respuestas totales: 2775


#### Preparación de los datos

Se conservan únicamente las preguntas con al menos dos respuestas candidatas y con alguna señal de preferencia, es decir, aquellas en las que al menos una respuesta tiene algún upvote.

In [4]:
df_modelo = df.copy()

# Preguntas con al menos dos respuestas
n_respuestas = (
    df_modelo.groupby("questionID")["answerText"]
    .transform("size")
)

df_modelo = df_modelo[
    n_respuestas >= 2
].copy()

# Preguntas con alguna señal de preferencia
max_upvotes = (
    df_modelo.groupby("questionID")["upvotes"]
    .transform("max")
)

df_modelo = df_modelo[
    max_upvotes > 0
].copy()

# Target: respuesta con máximo número de upvotes
df_modelo["target"] = (
    df_modelo["upvotes"]
    == df_modelo.groupby("questionID")["upvotes"]
    .transform("max")
).astype(int)

print(
    "Preguntas incluidas:",
    df_modelo["questionID"].nunique()
)

print(
    "Respuestas incluidas:",
    len(df_modelo)
)

Preguntas incluidas: 276
Respuestas incluidas: 1732


#### Caracterización de las preguntas

Para construir particiones más comparables, se resume la información a nivel de pregunta.
Para cada questionID se calcula:

- El número de respuestas candidatas.
- Cuántas respuestas comparten el máximo número de upvotes.
- si existe o no empate en la respuesta top.

In [5]:
info_preguntas = (
    df_modelo.groupby("questionID")
    .agg(
        n_respuestas=("answerText", "size")
    )
    .reset_index()
)

n_top = (
    df_modelo.assign(
        es_top=lambda x:
            x["upvotes"]
            == x.groupby("questionID")["upvotes"].transform("max")
    )
    .groupby("questionID")["es_top"]
    .sum()
    .reset_index(name="n_respuestas_top")
)

info_preguntas = info_preguntas.merge(
    n_top,
    on="questionID"
)

info_preguntas["empate"] = (
    info_preguntas["n_respuestas_top"] > 1
)

display(info_preguntas.head())

,questionID,n_respuestas,n_respuestas_top,empate
0,0,86,1,False
1,1,5,1,False
2,2,2,1,False
3,3,3,1,False
4,4,6,1,False


Comprobación rápida para detectar si algo salió raro:

In [6]:
print("Preguntas:", len(info_preguntas))
print(
    "Preguntas con empate:",
    info_preguntas["empate"].sum()
)
print(
    "Porcentaje con empate:",
    f"{info_preguntas['empate'].mean():.2%}"
)

Preguntas: 276
Preguntas con empate: 62
Porcentaje con empate: 22.46%


#### Definición de los estratos

Las preguntas se agrupan según dos características:

- El número de respuestas candidatas.
- La presencia o ausencia de empate en el máximo número de upvotes.

Para el número de respuestas se utilizan los intervalos 2, 3, 4-5, 6-10 y >10.
La combinación de ambas características define el estrato de cada pregunta.
Estos estratos se utilizarán posteriormente para intentar mantener una composición similar entre train, test y los folds de validación.

In [7]:
def grupo_n_respuestas(n):
    if n == 2:
        return "2"
    elif n == 3:
        return "3"
    elif n <= 5:
        return "4-5"
    elif n <= 10:
        return "6-10"
    else:
        return ">10"


info_preguntas["grupo_respuestas"] = (
    info_preguntas["n_respuestas"]
    .apply(grupo_n_respuestas)
)

info_preguntas["estrato"] = (
    info_preguntas["grupo_respuestas"]
    + ", empate="
    + info_preguntas["empate"].astype(str)
)

Se comprueba el número de preguntas de cada estrato antes de realizar las divisiones. Esto permite verificar que los grupos contienen suficientes ejemplos para poder distribuirlos entre las distintas particiones.

In [8]:
distribucion_estratos = (
    info_preguntas["estrato"]
    .value_counts()
    .sort_index()
)

display(
    distribucion_estratos.to_frame(
        "Número de preguntas"
    )
)

,Número de preguntas
estrato,
"2, empate=False",82
"2, empate=True",16
"3, empate=False",42
"3, empate=True",10
"4-5, empate=False",35
"4-5, empate=True",10
"6-10, empate=False",34
"6-10, empate=True",14
">10, empate=False",21


## Nuevo split train-test estratificado

Anteriormente, el split 80/20 se realizaba aleatoriamente por questionID. En esta versión se mantiene el mismo porcentaje, pero se realiza una división estratificada utilizando los estratos definidos anteriormente.
De esta forma se intenta mantener en train y test proporciones similares de:

- Preguntas con y sin empate.
- Preguntas con distinto número de respuestas candidatas.

La división continúa realizándose a nivel de questionID, por lo que todas las respuestas de una misma pregunta permanecen siempre en el mismo conjunto.

In [9]:
from sklearn.model_selection import train_test_split

preguntas_train, preguntas_test = train_test_split(
    info_preguntas,
    test_size=0.20,
    random_state=27,
    stratify=info_preguntas["estrato"]
)

Una vez asignadas las preguntas a train y test, se recuperan del dataset original todas las respuestas correspondientes a esas preguntas. Así, cada pregunta mantiene juntas todas sus respuestas dentro de una única partición.

In [10]:
train_equilibrado = df_modelo[
    df_modelo["questionID"].isin(
        preguntas_train["questionID"]
    )
].copy()

test_equilibrado = df_modelo[
    df_modelo["questionID"].isin(
        preguntas_test["questionID"]
    )
].copy()

print(
    "Preguntas en train:",
    train_equilibrado["questionID"].nunique()
)

print(
    "Preguntas en test:",
    test_equilibrado["questionID"].nunique()
)

Preguntas en train: 220
Preguntas en test: 56


In [11]:
preguntas_comunes = (
    set(train_equilibrado["questionID"])
    & set(test_equilibrado["questionID"])
)

print(
    "Preguntas compartidas entre train y test:",
    len(preguntas_comunes)
)

Preguntas compartidas entre train y test: 0


No existe solapamiento de preguntas entre train y test, por lo que todas las respuestas asociadas a un mismo questionID permanecen dentro de una única partición.

## Construcción de los 5 folds estratificados

Sobre el nuevo conjunto de train se construyen 5 folds de validación.
Es un cambio respecto al notebook anterior, donde se utilizaba GroupKFold únicamente para mantener juntas las respuestas de cada pregunta.
En esta versión, la división se realiza primero a nivel de pregunta mediante StratifiedKFold, utilizando el estrato como criterio. De esta forma, además de mantener las preguntas separadas, se intenta que cada fold tenga una proporción similar de empates y de números de respuestas candidatas.

In [12]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=27
)

folds_preguntas = list(
    skf.split(
        preguntas_train,
        preguntas_train["estrato"]
    )
)

print(
    "Número de folds:",
    len(folds_preguntas)
)

Número de folds: 5


Los 5 folds ya están definidos por pregunta. Ahora se localizan en train_equilibrado todas las respuestas correspondientes a las preguntas de entrenamiento y validación de cada fold.

In [13]:
cv_splits_equilibrados = []

for train_q_idx, val_q_idx in folds_preguntas:

    ids_train_fold = set(
        preguntas_train.iloc[
            train_q_idx
        ]["questionID"]
    )

    ids_val_fold = set(
        preguntas_train.iloc[
            val_q_idx
        ]["questionID"]
    )

    train_idx = np.where(
        train_equilibrado["questionID"]
        .isin(ids_train_fold)
    )[0]

    val_idx = np.where(
        train_equilibrado["questionID"]
        .isin(ids_val_fold)
    )[0]

    cv_splits_equilibrados.append(
        (train_idx, val_idx)
    )

Comprobación:

In [14]:
for fold, (train_idx, val_idx) in enumerate(
    cv_splits_equilibrados,
    start=1
):

    train_fold = train_equilibrado.iloc[
        train_idx
    ]

    val_fold = train_equilibrado.iloc[
        val_idx
    ]

    comunes = (
        set(train_fold["questionID"])
        & set(val_fold["questionID"])
    )

    print(
        f"Fold {fold}: "
        f"{train_fold['questionID'].nunique()} preguntas train | "
        f"{val_fold['questionID'].nunique()} preguntas validación | "
        f"{len(comunes)} compartidas"
    )

Fold 1: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 2: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 3: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 4: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 5: 176 preguntas train | 44 preguntas validación | 0 compartidas


Se comprueba que en los 5 folds hay 0 preguntas compartidas entre entrenamiento y validación.

## Comprobación de las nuevas particiones

Antes de repetir el entrenamiento de BERT, se analiza la composición de los nuevos folds de validación y del conjunto de test.
Se comprueba:
- El número de preguntas.
- La proporción de preguntas con empate.
- El número medio y mediano de respuestas candidatas.
- La distribución del número de respuestas por pregunta.

El objetivo es verificar que las nuevas particiones sean aproximadamente comparables antes de utilizar sus resultados para evaluar el modelo.

In [15]:
def resumen_particion(info, nombre):
    return {
        "Partición": nombre,
        "Preguntas": len(info),
        "Preguntas con empate": info["empate"].sum(),
        "% con empate": info["empate"].mean() * 100,
        "Media respuestas": info["n_respuestas"].mean(),
        "Mediana respuestas": info["n_respuestas"].median(),
        "Mín. respuestas": info["n_respuestas"].min(),
        "Máx. respuestas": info["n_respuestas"].max()
    }

In [16]:
resumen_nuevo = []

for fold, (_, val_q_idx) in enumerate(
    folds_preguntas,
    start=1
):

    info_val = preguntas_train.iloc[
        val_q_idx
    ]

    resumen_nuevo.append(
        resumen_particion(
            info_val,
            f"Fold {fold}"
        )
    )

resumen_nuevo.append(
    resumen_particion(
        preguntas_test,
        "Test"
    )
)

resumen_nuevo = pd.DataFrame(
    resumen_nuevo
)

display(
    resumen_nuevo.round(2)
)

,Partición,Preguntas,Preguntas con empate,% con empate,Media respuestas,Mediana respuestas,Mín. respuestas,Máx. respuestas
0,Fold 1,44,10,22.73,6.11,3.0,2,51
1,Fold 2,44,9,20.45,5.25,3.0,2,28
2,Fold 3,44,9,20.45,5.52,3.0,2,31
3,Fold 4,44,11,25.00,8.61,3.0,2,107
4,Fold 5,44,11,25.00,6.93,3.0,2,86
5,Test,56,12,21.43,5.45,3.0,2,35


**Número de preguntas:** Los cinco folds tienen exactamente 44 preguntas cada uno. Test tiene 56, lo cual es normal: cada fold corresponde aproximadamente a una quinta parte de train, mientras que test es el 20 % del dataset completo. Aquí ya no hay el pequeño desequilibrio anterior de 40–44 preguntas entre folds.

**Preguntas con empate:** Estas están mucho más alineadas. Antes los folds iban desde 14,63 % hasta 30,95 %, mientras que test tenía 25,93 %. Ahora todos están aproximadamente en el mismo rango.



#### Distribución del número de respuestas

In [17]:
distribuciones_nuevas = []

for fold, (_, val_q_idx) in enumerate(
    folds_preguntas,
    start=1
):

    info_val = preguntas_train.iloc[
        val_q_idx
    ]

    dist = (
        info_val["grupo_respuestas"]
        .value_counts(normalize=True)
        .mul(100)
    )

    dist.name = f"Fold {fold}"
    distribuciones_nuevas.append(dist)


dist_test = (
    preguntas_test["grupo_respuestas"]
    .value_counts(normalize=True)
    .mul(100)
)

dist_test.name = "Test"

distribuciones_nuevas.append(
    dist_test
)

tabla_distribucion = pd.concat(
    distribuciones_nuevas,
    axis=1
).fillna(0)

display(
    tabla_distribucion
    .reindex(
        ["2", "3", "4-5", "6-10", ">10"]
    )
    .round(2)
)

,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Test
grupo_respuestas,,,,,,
2,36.36,34.09,34.09,36.36,36.36,35.71
3,18.18,20.45,20.45,18.18,15.91,19.64
4-5,18.18,15.91,15.91,15.91,15.91,16.07
6-10,15.91,18.18,18.18,15.91,18.18,17.86
>10,11.36,11.36,11.36,13.64,13.64,10.71


La distribución del número de respuestas también ha mejorado mucho.

Por ejemplo, en preguntas con solo 2 respuestas:
- folds: 34,09–36,36 %
- test: 35,71 %

Con 4–5 respuestas:
- folds: 15,91–18,18 %
- test: 16,07 %

Con 6–10 respuestas:
- folds: 15,91–18,18 %
- test: 17,86 %

Y con más de 10 respuestas:
- folds: 11,36–13,64 %
- test: 10,71 %

Esto es especialmente importante porque antes test solo tenía 5,56 % de preguntas con más de 10 respuestas, mientras que los folds tenían entre 12,20 % y 17,07 %. Esa diferencia prácticamente ha desaparecido.

#### Distribución de los estratos

Esta comprobación es nueva y útil, porque permite ver conjuntamente las dos variables (número de respuestas candidatasy presencia o ausencia de empate en el máximo de upvotes) que hemos utilizado para equilibrar.

In [18]:
distribuciones_estratos = []

for fold, (_, val_q_idx) in enumerate(
    folds_preguntas,
    start=1
):

    info_val = preguntas_train.iloc[
        val_q_idx
    ]

    dist = (
        info_val["estrato"]
        .value_counts(normalize=True)
        .mul(100)
    )

    dist.name = f"Fold {fold}"
    distribuciones_estratos.append(dist)


dist_test_estratos = (
    preguntas_test["estrato"]
    .value_counts(normalize=True)
    .mul(100)
)

dist_test_estratos.name = "Test"

distribuciones_estratos.append(
    dist_test_estratos
)

tabla_estratos = pd.concat(
    distribuciones_estratos,
    axis=1
).fillna(0)

display(
    tabla_estratos
    .sort_index()
    .round(2)
)

,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Test
estrato,,,,,,
"2, empate=False",29.55,29.55,29.55,29.55,29.55,30.36
"2, empate=True",6.82,4.55,4.55,6.82,6.82,5.36
"3, empate=False",15.91,15.91,15.91,13.64,13.64,16.07
"3, empate=True",2.27,4.55,4.55,4.55,2.27,3.57
"4-5, empate=False",13.64,13.64,13.64,11.36,11.36,12.50
"4-5, empate=True",4.55,2.27,2.27,4.55,4.55,3.57
"6-10, empate=False",11.36,13.64,13.64,11.36,11.36,12.50
"6-10, empate=True",4.55,4.55,4.55,4.55,6.82,5.36
">10, empate=False",6.82,6.82,6.82,9.09,9.09,7.14


La distribución de los estratos es muy similar entre los cinco folds y test. Cada estrato combina el número de respuestas candidatas y la presencia o ausencia de empate, y sus proporciones se mantienen bastante próximas entre las distintas particiones.
Las pequeñas diferencias observadas son esperables debido al número limitado de preguntas en algunos estratos. En conjunto, la estratificación ha permitido obtener particiones mucho más comparables, por lo que resultan más adecuadas para repetir la validación cruzada de BERT.

## BERT pairwise con las nuevas particiones

Una vez comprobado que los nuevos folds y test presentan una composición aproximadamente comparable, se repite el entrenamiento del modelo BERT por pares.

Se mantiene el mismo procedimiento del experimento anterior para poder estudiar específicamente el efecto del nuevo particionado:
- Se comparan respuestas de una misma pregunta dos a dos.
- Los upvotes se utilizan únicamente para determinar cuál de las dos respuestas es preferida durante el entrenamiento.
- Los pares con el mismo número de upvotes se descartan.
- Cada par se presenta también en orden inverso para reducir el posible sesgo de posición.

Se mantiene bert-base-uncased y el límite de 512 tokens utilizado anteriormente. El posible efecto del truncado se analizará posteriormente de forma independiente.

#### Preparación de BERT

Se carga bert-base-uncased y se comprueba que PyTorch puede utilizar la GPU disponible.

In [19]:
import gc
import torch

from itertools import combinations
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)

set_seed(27)

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU disponible: False


In [20]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

#### Construcción de los pares de entrenamiento

Para cada pregunta se generan todas las combinaciones posibles de dos respuestas.

La respuesta con mayor número de upvotes se considera preferida:
- label = 0: la primera respuesta es preferida.
- label = 1: la segunda respuesta es preferida.

Si ambas tienen los mismos upvotes, el par se descarta porque no existe una preferencia observable entre ellas.

Cada comparación se añade también en orden inverso.

In [21]:
def crear_pares_entrenamiento(datos):
    ejemplos = []

    for question_id, grupo in datos.groupby("questionID"):

        filas = list(
            grupo.itertuples(index=False)
        )

        for respuesta_1, respuesta_2 in combinations(
            filas, 2
        ):

            # No existe una preferencia clara
            if respuesta_1.upvotes == respuesta_2.upvotes:
                continue

            pregunta = str(
                respuesta_1.questionText
            )

            texto_1 = str(
                respuesta_1.answerText
            )

            texto_2 = str(
                respuesta_2.answerText
            )

            if respuesta_1.upvotes > respuesta_2.upvotes:
                label = 0
            else:
                label = 1

            # Orden A-B
            ejemplos.append({
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_1}"
                ),
                "texto_b": (
                    f"Response B: {texto_2}"
                ),
                "label": label
            })

            # Orden B-A
            ejemplos.append({
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_2}"
                ),
                "texto_b": (
                    f"Response B: {texto_1}"
                ),
                "label": 1 - label
            })

    return pd.DataFrame(ejemplos)

#### Preparación de los datos para BERT

Los textos de cada par se tokenizan para transformarlos al formato que necesita BERT. Las secuencias que superan los 512 tokens se truncan, manteniendo por ahora la misma configuración que en el experimento anterior.

In [22]:
def preparar_dataset_bert(
    pares,
    entrenamiento=True
):

    columnas = [
        "texto_a",
        "texto_b"
    ]

    if entrenamiento:
        columnas.append("label")

    dataset = Dataset.from_pandas(
        pares[columnas],
        preserve_index=False
    )

    def tokenizar(batch):
        return tokenizer(
            batch["texto_a"],
            batch["texto_b"],
            truncation=True,
            max_length=512
        )

    dataset = dataset.map(
        tokenizar,
        batched=True
    )

    dataset = dataset.remove_columns(
        ["texto_a", "texto_b"]
    )

    if entrenamiento:
        dataset = dataset.rename_column(
            "label",
            "labels"
        )

    return dataset

#### Análisis del límite de 512 tokens

Antes del entrenamiento se comprueba qué proporción de los pares supera el límite de 512 tokens de bert-base-uncased.
Estos casos se truncarán durante la tokenización, por lo que parte del texto no llegará al modelo. El objetivo es medir primero la magnitud del problema antes de valorar otras estrategias o modelos capaces de procesar secuencias más largas.

In [23]:
def longitud_tokens_pares(pares):
    longitudes = []

    for texto_a, texto_b in zip(
        pares["texto_a"],
        pares["texto_b"]
    ):
        tokens = tokenizer(
            texto_a,
            texto_b,
            truncation=False
        )

        longitudes.append(
            len(tokens["input_ids"])
        )

    return np.array(longitudes)

A continuación, se generan los pares de entrenamiento y se calcula la longitud en tokens de cada uno.

In [24]:
pares_analisis = crear_pares_entrenamiento(
    train_equilibrado
)

longitudes = longitud_tokens_pares(
    pares_analisis
)

print("Número de ejemplos (parejas en ambos órdenes):", len(longitudes))
print(
    "Ejemplos con más de 512 tokens:",
    (longitudes > 512).sum()
)
print(
    "Porcentaje truncado:",
    f"{(longitudes > 512).mean():.2%}"
)
print(
    "Longitud mediana:",
    np.median(longitudes)
)
print(
    "Longitud máxima:",
    longitudes.max()
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (626 > 512). Running this sequence through the model will result in indexing errors


Número de ejemplos (parejas en ambos órdenes): 8902
Ejemplos con más de 512 tokens: 2612
Porcentaje truncado: 29.34%
Longitud mediana: 411.0
Longitud máxima: 1638


De los 8.902 ejemplos de entrenamiento (4.451 parejas en ambos órdenes), 2.612 (29,34 %) superan el límite de 512 tokens de BERT. La longitud mediana es de 411 tokens y la máxima alcanza 1.638 tokens. Por tanto, el truncado afecta a una proporción relevante de los ejemplos y podría provocar pérdida de información en algunas comparaciones.
Dado que las particiones ya han sido revisadas y equilibradas, este resultado justifica estudiar posteriormente una estrategia que permita procesar secuencias más largas, como ModernBERT, y comparar su rendimiento con bert-base-uncased.

#### Selección de la respuesta top-1

Durante la evaluación, BERT compara todas las respuestas de una pregunta dos a dos y asigna una probabilidad de preferencia a cada una. Cada respuesta obtiene una victoria cuando es elegida frente a otra.
La respuesta con más victorias se selecciona como top-1. Si varias respuestas obtienen el mismo número de victorias, se elige la que haya acumulado una mayor probabilidad de preferencia en sus comparaciones.
En esta fase, BERT recibe únicamente los textos. Los upvotes se utilizaron anteriormente para crear las etiquetas de entrenamiento y se usan después, mediante el target, para comprobar si la respuesta seleccionada es correcta.

In [ ]:
def crear_pares_prediccion(datos):

    datos_eval = (
        datos.reset_index(drop=True)
        .copy()
    )

    datos_eval["row_id"] = np.arange(
        len(datos_eval)
    )

    ejemplos = []
    pair_id = 0

    for question_id, grupo in datos_eval.groupby(
        "questionID"
    ):

        filas = list(
            grupo.itertuples(index=False)
        )

        for respuesta_1, respuesta_2 in combinations(
            filas, 2
        ):

            pregunta = str(
                respuesta_1.questionText
            )

            texto_1 = str(
                respuesta_1.answerText
            )

            texto_2 = str(
                respuesta_2.answerText
            )

            # Orden A-B
            ejemplos.append({
                "pair_id": pair_id,
                "questionID": question_id,
                "row_i": respuesta_1.row_id,
                "row_j": respuesta_2.row_id,
                "orden": "ij",
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_1}"
                ),
                "texto_b": (
                    f"Response B: {texto_2}"
                )
            })

            # Orden B-A
            ejemplos.append({
                "pair_id": pair_id,
                "questionID": question_id,
                "row_i": respuesta_1.row_id,
                "row_j": respuesta_2.row_id,
                "orden": "ji",
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_2}"
                ),
                "texto_b": (
                    f"Response B: {texto_1}"
                )
            })

            pair_id += 1

    return (
        datos_eval,
        pd.DataFrame(ejemplos)
    )

La siguiente función compara las respuestas, selecciona la respuesta top-1 de cada pregunta y, finalmente, utiliza el target, definido previamente a partir de los upvotes, para comprobar si la predicción fue correcta y calcular Accuracy@1.

In [ ]:
def evaluar_torneo(trainer, datos):

    datos_eval, pares = (
        crear_pares_prediccion(datos)
    )

    dataset = preparar_dataset_bert(
        pares,
        entrenamiento=False
    )

    predicciones = trainer.predict(
        dataset
    )

    logits = torch.tensor(
        predicciones.predictions
    )

    probabilidades = torch.softmax(
        logits,
        dim=1
    ).numpy()

    pares["prob_primera"] = (
        probabilidades[:, 0]
    )

    pares["prob_segunda"] = (
        probabilidades[:, 1]
    )

    victorias = {
        row_id: 0.0
        for row_id in datos_eval["row_id"]
    }

    puntuaciones = {
        row_id: 0.0
        for row_id in datos_eval["row_id"]
    }

    for _, grupo in pares.groupby(
        "pair_id"
    ):

        orden_ij = grupo[
            grupo["orden"] == "ij"
        ].iloc[0]

        orden_ji = grupo[
            grupo["orden"] == "ji"
        ].iloc[0]

        row_i = orden_ij["row_i"]
        row_j = orden_ij["row_j"]

        prob_i = (
            orden_ij["prob_primera"]
            + orden_ji["prob_segunda"]
        ) / 2

        prob_j = 1 - prob_i

        puntuaciones[row_i] += prob_i
        puntuaciones[row_j] += prob_j

        if prob_i > prob_j:
            victorias[row_i] += 1

        elif prob_j > prob_i:
            victorias[row_j] += 1

        else:
            victorias[row_i] += 0.5
            victorias[row_j] += 0.5

    ganadores = []

    for question_id, grupo in datos_eval.groupby(
        "questionID"
    ):

        grupo = grupo.copy()

        grupo["victorias"] = (
            grupo["row_id"]
            .map(victorias)
        )

        grupo["puntuacion"] = (
            grupo["row_id"]
            .map(puntuaciones)
        )

        ganador = (
            grupo.sort_values(
                ["victorias", "puntuacion"],
                ascending=[False, False]
            )
            .iloc[0]
        )

        ganadores.append(ganador)

    ganadores = pd.DataFrame(
        ganadores
    )

    accuracy = (
        ganadores["target"].mean()
    )

    return accuracy, ganadores

#### Validación cruzada con los nuevos folds

El modelo se entrena y evalúa utilizando los 5 folds estratificados definidos anteriormente.

En cada iteración:

- 4 folds se utilizan para entrenar BERT.
- 1 fold se utiliza para validación.
- Los pares empatados en upvotes se descartan únicamente del entrenamiento.
- Todas las preguntas del fold de validación se mantienen.
- Se calcula Accuracy@1 únicamente sobre dicho fold.

El proceso se repite cinco veces y posteriormente se calcula la media y la desviación estándar de los resultados.

In [ ]:
accuracies_bert_equilibrado_cv = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits_equilibrados,
    start=1
):

    print(f"\n===== FOLD {fold} =====")

    train_fold = (
        train_equilibrado.iloc[
            train_idx
        ].copy()
    )

    val_fold = (
        train_equilibrado.iloc[
            val_idx
        ].copy()
    )

    pares_fold = crear_pares_entrenamiento(
        train_fold
    )

    print(
        "Preguntas train:",
        train_fold["questionID"].nunique()
    )

    print(
        "Preguntas validación:",
        val_fold["questionID"].nunique()
    )

    print(
        "Pares de entrenamiento:",
        len(pares_fold)
    )

    dataset_fold = preparar_dataset_bert(
        pares_fold,
        entrenamiento=True
    )

    modelo_fold = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME,
            num_labels=2
        )
    )

    argumentos_fold = TrainingArguments(
        output_dir=(
            f"./outputs/"
            f"bert_equilibrado_fold_{fold}"
        ),
        num_train_epochs=2,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
        seed=27,
        data_seed=27
    )

    trainer_fold = Trainer(
        model=modelo_fold,
        args=argumentos_fold,
        train_dataset=dataset_fold,
        data_collator=data_collator
    )

    trainer_fold.train()

    accuracy_fold, _ = evaluar_torneo(
        trainer_fold,
        val_fold
    )

    accuracies_bert_equilibrado_cv.append(
        accuracy_fold
    )

    print(
        f"Fold {fold} - Accuracy@1: "
        f"{accuracy_fold:.2%}"
    )

    del trainer_fold
    del modelo_fold
    del dataset_fold

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


===== FOLD 1 =====
Preguntas train: 176
Preguntas validación: 44
Pares de entrenamiento: 7982


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 3191.02it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

Step,Training Loss
998,0.332737
1996,0.070373


Map: 100%|████████████████████████| 4378/4378 [00:01<00:00, 4033.06 examples/s]


Fold 1 - Accuracy@1: 45.45%

===== FOLD 2 =====
Preguntas train: 176
Preguntas validación: 44
Pares de entrenamiento: 8024


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 2444.43it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

... (11 outputs left out)

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 2612.99it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

Step,Training Loss
764,0.356862
1528,0.079786


Map: 100%|████████████████████████| 9072/9072 [00:01<00:00, 5098.63 examples/s]


Fold 5 - Accuracy@1: 36.36%


Resumen de la validación cruzada:

In [ ]:
print(
    "Accuracy@1 media CV:",
    f"{np.mean(accuracies_bert_equilibrado_cv):.2%}"
)

print(
    "Desviación estándar:",
    f"{np.std(accuracies_bert_equilibrado_cv):.2%}"
)

Accuracy@1 media CV: 44.09%
Desviación estándar: 5.10%


La validación cruzada obtiene una Accuracy@1 media del 44,09 %, frente al 45,00 % del experimento anterior, una diferencia pequeña de 0,91 puntos porcentuales. La desviación estándar aumenta de 2,76 % a 5,10 %, lo que indica una mayor variación entre folds. Por tanto, el nuevo particionado no mejora el rendimiento de BERT, aunque permite realizar la validación sobre folds con una composición más comparable en número de respuestas y empates.

#### Entrenamiento final con todo train

Tras la validación cruzada, se entrena un nuevo modelo utilizando todas las preguntas del conjunto train_equilibrado. Como en los folds anteriores, se conservan las preguntas con empate y únicamente se descartan los pares concretos cuyas dos respuestas tienen el mismo número de upvotes.

In [25]:
pares_train_final = (
    crear_pares_entrenamiento(
        train_equilibrado
    )
)

print(
    "Preguntas de entrenamiento:",
    train_equilibrado[
        "questionID"
    ].nunique()
)

print(
    "Ejemplos de entrenamiento (parejas en ambos órdenes):",
    len(pares_train_final)
)

Preguntas de entrenamiento: 220
Ejemplos de entrenamiento (parejas en ambos órdenes): 8902


In [ ]:
dataset_train_final = (
    preparar_dataset_bert(
        pares_train_final,
        entrenamiento=True
    )
)

modelo_final = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
)

argumentos_finales = TrainingArguments(
    output_dir=(
        "./outputs/"
        "bert_equilibrado_final"
    ),
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=27,
    data_seed=27
)

trainer_final = Trainer(
    model=modelo_final,
    args=argumentos_finales,
    train_dataset=dataset_train_final,
    data_collator=data_collator
)

trainer_final.train()

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 2411.97it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

Step,Training Loss
1113,0.376987
2226,0.072382


TrainOutput(global_step=2226, training_loss=0.22468468185383675, metrics={'train_runtime': 433.359, 'train_samples_per_second': 41.084, 'train_steps_per_second': 5.137, 'total_flos': 4662940456404960.0, 'train_loss': 0.22468468185383675, 'epoch': 2.0})

#### Evaluación final en test

Finalmente, el modelo entrenado con todo train_equilibrado se evalúa sobre test_equilibrado.

BERT compara todas las respuestas candidatas de cada pregunta dos a dos y se selecciona como top-1 la respuesta con mayor número de victorias.
En las preguntas con empate en el máximo número de upvotes, cualquiera de las respuestas top se considera correcta.

In [ ]:
accuracy_bert_equilibrado_test, ganadores_finales = (
    evaluar_torneo(
        trainer_final,
        test_equilibrado
    )
)

print(
    "BERT pairwise con particiones equilibradas "
    "- Accuracy@1 test:",
    f"{accuracy_bert_equilibrado_test:.2%}"
)

Map: 100%|████████████████████████| 3238/3238 [00:00<00:00, 3568.34 examples/s]


BERT pairwise con particiones equilibradas - Accuracy@1 test: 37.50%


Tabla resumen:

In [ ]:
resultados_bert_equilibrado = pd.DataFrame({
    "Métrica": [
        "Accuracy@1 media CV",
        "Desviación estándar CV",
        "Accuracy@1 test"
    ],
    "Resultado (%)": [
        np.mean(
            accuracies_bert_equilibrado_cv
        ) * 100,

        np.std(
            accuracies_bert_equilibrado_cv
        ) * 100,

        accuracy_bert_equilibrado_test * 100
    ]
})

display(
    resultados_bert_equilibrado.round(2)
)

,Métrica,Resultado (%)
0,Accuracy@1 media CV,44.09
1,Desviación estándar CV,5.10
2,Accuracy@1 test,37.50


En el conjunto de test, el modelo obtiene una Accuracy@1 del 37,50 %, frente al 57,14 % obtenido anteriormente. Sin embargo, ambos resultados proceden de conjuntos de test distintos, por lo que no pueden compararse directamente. En el nuevo experimento, el resultado de test queda 6,59 puntos porcentuales por debajo de la media de validación (44,09 %), mostrando un rendimiento inferior al esperado sobre datos no utilizados durante el entrenamiento.